In [2]:
import torch

class IrisNeuralNetwork:
    def __init__(self, input_size, hidden1_size, hidden2_size):
        # Weight initialization has to be appropriate for the 
        # architecture and activation functions.
        self.w12 = torch.randn(input_size, hidden1_size)
        self.b1 = torch.zeros(hidden1_size)
        self.w23 = torch.randn(hidden1_size, hidden2_size)
        self.b2 = torch.zeros(hidden2_size)
        self.w34 = torch.randn(hidden2_size, 1)
        self.b3 = torch.zeros(1)

    def sigmoid(self, x):
        return 1 / (1 + torch.exp(-x))

    def forward(self, x):
        nc = nw = 0
        # Forward pass
        z1 = x @ self.w12 + self.b1
        a1 = self.sigmoid(z1)
        z2 = a1 @ self.w23 + self.b2
        a2 = self.sigmoid(z2)
        z3 = a2 @ self.w34 + self.b3
        a3 = self.sigmoid(z3)

        return a3;   

# Load data

x_train = torch.load(
    "../data/train/iris2_train.pt"
).float()

y_train = torch.load(
    "../data/train/iris2_train_labels.pt"
).float().reshape(-1, 1)


x_test = torch.load("../data/test/iris2_test.pt").float()
y_test = torch.load("../data/test/iris2_test_labels.pt").float().reshape(-1, 1)

model = IrisNeuralNetwork(
    input_size=2,
    hidden1_size=3,
    hidden2_size=2
)

print(model.w12)
print(model.b1)
print(model.w23)
print(model.b2)
print(model.w34)
print(model.b3)
# weights = pickle.load(open("../data/weights/iris2_weights.pkl", "rb"))
# model = IrisNeuralNetwork(input_size=2, hidden1_size=8, hidden2_size=8)
# probabilities = model.forward(x_test)
# print(probabilities)

tensor([[-0.9215,  0.6023,  1.0357],
        [ 1.9561, -0.7656, -0.3006]])
tensor([0., 0., 0.])
tensor([[-0.7572,  0.7287],
        [-0.2396,  0.7205],
        [-0.8325, -0.3777]])
tensor([0., 0.])
tensor([[-0.2526],
        [-1.0750]])
tensor([0.])


In [3]:
def binary_cross_entropy(y, prediction):

    # Prevent log(0)
    epsilon = 1e-7

    prediction = torch.clamp(
        prediction,
        epsilon,
        1 - epsilon
    )

    loss = -(
        y * torch.log(prediction)
        +
        (1 - y) * torch.log(1 - prediction)
    )

    return loss.mean()

def backward(model, x, y, prediction):

    # Number of samples
    m = x.shape[0]

    # -----------------------------------------
    # Recalculate forward-pass intermediates
    # -----------------------------------------

    z1 = x @ model.w12 + model.b1
    a1 = model.sigmoid(z1)

    z2 = a1 @ model.w23 + model.b2
    a2 = model.sigmoid(z2)

    z3 = a2 @ model.w34 + model.b3
    a3 = prediction

    # -----------------------------------------
    # Output layer
    # -----------------------------------------

    # For sigmoid + binary cross entropy:
    # dL/dz3 = a3 - y
    dz3 = a3 - y

    dw34 = (a2.T @ dz3) / m
    db3 = dz3.mean(dim=0)

    # -----------------------------------------
    # Hidden layer 2
    # -----------------------------------------

    da2 = dz3 @ model.w34.T

    dz2 = da2 * a2 * (1 - a2)

    dw23 = (a1.T @ dz2) / m
    db2 = dz2.mean(dim=0)

    # -----------------------------------------
    # Hidden layer 1
    # -----------------------------------------

    da1 = dz2 @ model.w23.T

    dz1 = da1 * a1 * (1 - a1)

    dw12 = (x.T @ dz1) / m
    db1 = dz1.mean(dim=0)

    return dw12, db1, dw23, db2, dw34, db3


def update_parameters(model, gradients, learning_rate):

    dw12, db1, dw23, db2, dw34, db3 = gradients

    model.w12 -= learning_rate * dw12
    model.b1  -= learning_rate * db1

    model.w23 -= learning_rate * dw23
    model.b2  -= learning_rate * db2

    model.w34 -= learning_rate * dw34
    model.b3  -= learning_rate * db3

In [4]:
def train(model, x_train, y_train, epochs, learning_rate):

    for epoch in range(epochs):

        # -----------------------------
        # 1. Forward pass
        # -----------------------------
        prediction = model.forward(x_train)

        # -----------------------------
        # 2. Calculate loss
        # -----------------------------
        loss = binary_cross_entropy(
            y_train,
            prediction
        )

        # -----------------------------
        # 3. Backpropagation
        # -----------------------------
        gradients = backward(
            model,
            x_train,
            y_train,
            prediction
        )

        # -----------------------------
        # 4. Gradient descent
        # -----------------------------
        update_parameters(
            model,
            gradients,
            learning_rate
        )

        # Print progress
        if epoch % 100 == 0:
            print(
                f"Epoch {epoch:4d} | "
                f"Loss: {loss.item():.6f}"
            )

In [5]:
# -----------------------------------------
# Train
# -----------------------------------------

train(
    model,
    x_train,
    y_train,
    epochs=5000,
    learning_rate=0.1
)


# -----------------------------------------
# Test
# -----------------------------------------

with torch.no_grad():

    probabilities = model.forward(x_test)

    predictions = (probabilities >= 0.5).float()

    accuracy = (
        predictions == y_test
    ).float().mean()


print()
print("Test results:")
print(f"Overall score: {accuracy.item():.7f}")

print()

for i in range(len(y_test)):

    actual = int(y_test[i].item())
    predicted = int(predictions[i].item())
    probability = probabilities[i].item()

    print(
        f"{i:03d}: "
        f"actual: {actual} "
        f"predict: {predicted} "
        f"prob: {probability:.7f}"
    )
print("\n\n")
print(model.w12)
print(model.b1)
print(model.w23)
print(model.b2)
print(model.w34)
print(model.b3)

Epoch    0 | Loss: 0.760736
Epoch  100 | Loss: 0.693404
Epoch  200 | Loss: 0.692405
Epoch  300 | Loss: 0.691027
Epoch  400 | Loss: 0.689578
Epoch  500 | Loss: 0.688462
Epoch  600 | Loss: 0.687492
Epoch  700 | Loss: 0.686463
Epoch  800 | Loss: 0.685266
Epoch  900 | Loss: 0.683824
Epoch 1000 | Loss: 0.682055
Epoch 1100 | Loss: 0.679854
Epoch 1200 | Loss: 0.677078
Epoch 1300 | Loss: 0.673536
Epoch 1400 | Loss: 0.668972
Epoch 1500 | Loss: 0.663055
Epoch 1600 | Loss: 0.655337
Epoch 1700 | Loss: 0.645160
Epoch 1800 | Loss: 0.631601
Epoch 1900 | Loss: 0.613523
Epoch 2000 | Loss: 0.589737
Epoch 2100 | Loss: 0.559407
Epoch 2200 | Loss: 0.522695
Epoch 2300 | Loss: 0.481319
Epoch 2400 | Loss: 0.438377
Epoch 2500 | Loss: 0.397246
Epoch 2600 | Loss: 0.360353
Epoch 2700 | Loss: 0.328721
Epoch 2800 | Loss: 0.302261
Epoch 2900 | Loss: 0.280318
Epoch 3000 | Loss: 0.262075
Epoch 3100 | Loss: 0.246772
Epoch 3200 | Loss: 0.233784
Epoch 3300 | Loss: 0.222627
Epoch 3400 | Loss: 0.212942
Epoch 3500 | Loss: 0

In [8]:
trained_weights = {
    "w12": model.w12,
    "b1": model.b1,
    "w23": model.w23,
    "b2": model.b2,
    "w34": model.w34,
    "b3": model.b3
}

torch.save(trained_weights, "../data/weights/iris_nn_weights.pt")

In [10]:
torch.save(model, "../models/iris_nn_model.pt")